In [ ]:
import os
import qdrant_client
import shutil
import importlib

from docling.datamodel.document import DoclingDocument
from llama_index.core import (
    TreeIndex,  
    VectorStoreIndex,
    StorageContext,
    Settings,
    Document,
)
from llama_index.core.node_parser import SentenceSplitter 
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai_like import OpenAILike
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.storage.docstore import SimpleDocumentStore
from qdrant_client.models import VectorParams, Distance, SparseVectorParams


collectionname = "WAMAS_TREE_INDEX"
url_embedder = os.getenv("VLLM_API_BASE_URL")
url_qdrant = os.getenv("QDRANT_URL")
url_llm = os.getenv("VLLM_API_BASE_URL")


client = qdrant_client.QdrantClient(url=url_qdrant)


embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key=url_embedder,
)
Settings.embed_model = embed_model


llm = OpenAILike(
    model='meta-llama/Llama-3.1-8B-Instruct',
    api_base=url_llm, 
    api_key="null",
    is_chat_model=True,
    timeout=60.0,
    context_window=4096,
    temperature=0,
)
Settings.llm = llm


vector_store = QdrantVectorStore(
    client=client,
    collection_name=collectionname,
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)


if client.collection_exists(collectionname):
    client.delete_collection(collectionname)

client.create_collection(
    collection_name=collectionname,
    vectors_config={
        "bge_m3": VectorParams(size=1024, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams()
    }
)

docstore = SimpleDocumentStore()
storage_context = StorageContext.from_defaults(vector_store=vector_store, docstore=docstore)


base_folder = "../preprocessing/scratch"
all_nodes = []

print("--- INIZIO ELABORAZIONE FILE ---")

for root, dirs, files in os.walk(base_folder):
    json_file = None
    md_file = None

    for file in files:
        if file.endswith(".json"):
            json_file = os.path.join(root, file)
        elif file.endswith(".md"):
            md_file = os.path.join(root, file)

    if json_file and md_file:
        try:
            
            print(f"Processando: {os.path.basename(json_file)}...", end=" ")
            doc = DoclingDocument.load_from_json(json_file)
            full_text = doc.export_to_markdown()
            
            llama_doc = Document(text=full_text, metadata={"filename": json_file})
            
            parser = SentenceSplitter(chunk_size=1024, chunk_overlap=20)
            file_nodes = parser.get_nodes_from_documents([llama_doc])
            
            
            all_nodes.extend(file_nodes)
            print(f"-> Estratti {len(file_nodes)} nodi.")
            
        except Exception as e:
            print(f"\nERRORE su {json_file}: {e}")


print(f"\n--- CREAZIONE TREE INDEX SU {len(all_nodes)} NODI TOTALI ---")
print("Questa fase userà l'LLM per riassumere e costruire la gerarchia...")

if len(all_nodes) > 0:

    storage_context.docstore.add_documents(all_nodes)


    index = TreeIndex(
        all_nodes,
        storage_context=storage_context,
        show_progress=True,
        build_tree=True 
    )


    persist_dir = "./storage_tree"
    storage_context.persist(persist_dir=persist_dir)
    print(f"Finito! Indice salvato in {persist_dir}")
else:
    print("Nessun nodo trovato. Controlla i percorsi.")

--- INIZIO ELABORAZIONE FILE ---
Processando: UTL_Refresh_Reset_OT.json... -> Estratti 3 nodi.
Processando: PUB_Creazione_nuove_UDC.json... -> Estratti 3 nodi.
Processando: UTL-MAN_Gestione_vuoti_errore_in_baia.json... -> Estratti 2 nodi.
Processando: UTL-MAN_Cambio_pinza.json... -> Estratti 3 nodi.
Processando: UTL_Gestione_vuoti_weekend.json... -> Estratti 2 nodi.
Processando: UTL-MAN_SETUP_AGV.json... -> Estratti 4 nodi.
Processando: PUB_Processo_tonalizzazione.json... -> Estratti 4 nodi.
Processando: UTL_Cambio_setup_AGV.json... -> Estratti 2 nodi.
Processando: UTL-MAN_Modifica_terminale_baie.json... -> Estratti 2 nodi.
Processando: UTL_Cambio_priorita_MAV2_S46.json... -> Estratti 2 nodi.
Processando: UTL_Assegnazioni_ordini_utente.json... -> Estratti 2 nodi.
Processando: UTL-MAN_Disconnessione_utenti_baie.json... -> Estratti 1 nodi.
Processando: UTL-MAN_Arresto_baie_picking.json... -> Estratti 3 nodi.
Processando: UTL_Rimozione_manuale_PLC_SOC.json... -> Estratti 2 nodi.
Processan

Generating summaries:   0%|          | 0/4 [00:00<?, ?it/s]

Finito! Indice salvato in ./storage_tree


Retrieve semplice

In [12]:
import os
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.vector_stores import SimpleVectorStore



url_embedder = os.getenv("VLLM_API_BASE_URL")
url_llm = os.getenv("VLLM_API_BASE_URL")


Settings.embed_model = OpenAIEmbedding(
    api_base=url_embedder,
    model_name="BAAI/bge-m3",
    api_key="null",
)


Settings.llm = OpenAILike(
    model='meta-llama/Llama-3.1-8B-Instruct',
    api_base=url_llm, 
    api_key="null",
    is_chat_model=True,
    context_window=4096,
    temperature=0.1,
)


persist_dir = "./storage_tree"
print(f"Caricamento indice da {persist_dir}...")

# errore del cazzo
storage_context = StorageContext.from_defaults(
    persist_dir=persist_dir,
    vector_store=SimpleVectorStore()
)

index = load_index_from_storage(storage_context)



query_engine = index.as_query_engine(
    retriever_mode="select_leaf_embedding", 
    response_mode="tree_summarize",
    verbose=True, 
    child_branch_factor=3, 
)


print("\n--- SISTEMA PRONTO. Scrivi 'exit' per uscire. ---")

while True:
    question = input("\nFai una domanda sui documenti: ")
    if question.lower() in ["exit"]:
        break
    
    if not question.strip():
        continue

    print("Sto ragionando...\n")
    try:
        response = query_engine.query(question)
        print(f"\nRISPOSTA:\n{response}")
        
        
        # print("\nFONTI USATE:")
        # for node in response.source_nodes:
        #     print("\nFONTE:")
        #     print(f"- {node.node.get_content()}...")
            
    except Exception as e:
        print(f"Errore nella query: {e}")

Caricamento indice da ./storage_tree...

--- SISTEMA PRONTO. Scrivi 'exit' per uscire. ---
Sto ragionando...

1 text chunks after repacking

RISPOSTA:
Per creare una nuova unità di carico, si parte dalla pagina SM023 su WAMAS. Si clicca tasto destro in un'area vuota e cliccare 'Nuova'. Si aprirà così la finestra SM028, nella vista 'Unità di carico' andranno compilati i seguenti campi:

- Unità di carico: inserire il codice della UDC che si vuole creare
- Posto di stoccaggio:
  - o 'Magazzino': cliccare sulla lente di ricerca e selezionare l'unica opzione disponibile
  - o 'Posto di stoccaggio': selezionare il posto di stoccaggio in cui si trova l'UDC fisica
- Identificazione del posto di stoccaggio: si compilerà automaticamente compilando la cella 'Posto di stoccaggio'
- Base di carico: selezionare il tipo di pallet utilizzato per l'UDC
- Cubatura della base di carico: selezionare l'altezza dell'UDC
- Grafo flusso merci: selezionare l'opzione desiderata

Una volta compilati questi camp